In [ ]:
import json
import os

# Get the base directory - works for both Colab and local
try:
    # For Google Colab
    get_ipython()
    base_dir = "/content/Graphcare"
    print(f"Running in Colab, base_dir: {base_dir}")
except:
    # For local execution
    base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in globals() else '.', '..', '..', '..'))
    print(f"Running locally, base_dir: {base_dir}")

file_dir = os.path.join(base_dir, "graphs", "condition", "CCSCM")

file_id2ent = os.path.join(file_dir, "id2ent.json")
file_ent2id = os.path.join(file_dir, "ent2id.json")
file_id2rel = os.path.join(file_dir, "id2rel.json")
file_rel2id = os.path.join(file_dir, "rel2id.json")

with open(file_id2ent, 'r') as file:
    cond_id2ent = json.load(file)
with open(file_ent2id, 'r') as file:
    cond_ent2id = json.load(file)
with open(file_id2rel, 'r') as file:
    cond_id2rel = json.load(file)
with open(file_rel2id, 'r') as file:
    cond_rel2id = json.load(file)


file_dir = os.path.join(base_dir, "graphs", "procedure", "CCSPROC")

file_id2ent = os.path.join(file_dir, "id2ent.json")
file_ent2id = os.path.join(file_dir, "ent2id.json")
file_id2rel = os.path.join(file_dir, "id2rel.json")
file_rel2id = os.path.join(file_dir, "rel2id.json")

with open(file_id2ent, 'r') as file:
    proc_id2ent = json.load(file)
with open(file_ent2id, 'r') as file:
    proc_ent2id = json.load(file)
with open(file_id2rel, 'r') as file:
    proc_id2rel = json.load(file)
with open(file_rel2id, 'r') as file:
    proc_rel2id = json.load(file)


file_dir = os.path.join(base_dir, "graphs", "drug", "ATC3")

file_id2ent = os.path.join(file_dir, "id2ent.json")
file_ent2id = os.path.join(file_dir, "ent2id.json")
file_id2rel = os.path.join(file_dir, "id2rel.json")
file_rel2id = os.path.join(file_dir, "rel2id.json")

with open(file_id2ent, 'r') as file:
    drug_id2ent = json.load(file)
with open(file_ent2id, 'r') as file:
    drug_ent2id = json.load(file)
with open(file_id2rel, 'r') as file:
    drug_id2rel = json.load(file)
with open(file_rel2id, 'r') as file:
    drug_rel2id = json.load(file)


import csv

condition_mapping_file = os.path.join(base_dir, "resources", "CCSCM.csv")
procedure_mapping_file = os.path.join(base_dir, "resources", "CCSPROC.csv")
drug_file = os.path.join(base_dir, "resources", "ATC.csv")

condition_dict = {}
with open(condition_mapping_file, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        condition_dict[row['code']] = row['name'].lower()

procedure_dict = {}
with open(procedure_mapping_file, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        procedure_dict[row['code']] = row['name'].lower()

drug_dict = {}
with open(drug_file, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        if row['level'] == '3.0':
            drug_dict[row['code']] = row['name'].lower()

In [2]:
cpd_id2ent = {}
cpd_id2rel = {}

for key in cond_id2ent.keys():
    cpd_id2ent[key] = cond_id2ent[key]

for key in proc_id2ent.keys():
    cpd_id2ent[str(int(key)+len(cond_id2ent))] = proc_id2ent[key]

for key in drug_id2ent.keys():
    cpd_id2ent[str(int(key)+len(cond_id2ent)+len(proc_id2ent))] = drug_id2ent[key]

cpd_ent2id = {value: key for key, value in cpd_id2ent.items()}


for key in cond_id2rel.keys():
    cpd_id2rel[key] = cond_id2rel[key]

for key in proc_id2rel.keys():
    cpd_id2rel[str(int(key)+len(cond_id2rel))] = proc_id2rel[key]

for key in drug_id2rel.keys():
    cpd_id2rel[str(int(key)+len(cond_id2rel)+len(proc_id2rel))] = drug_id2rel[key]

cpd_rel2id = {value: key for key, value in cpd_id2rel.items()}

In [3]:
len(cpd_id2ent), len(cpd_id2rel)

(42056, 9404)

In [ ]:
output_dir = os.path.join(base_dir, "graphs", "cond_proc_drug", "CCSCM_CCSPROC_ATC3")
os.makedirs(output_dir, exist_ok=True)

out_file_id2ent = os.path.join(output_dir, "id2ent.json")
out_file_ent2id = os.path.join(output_dir, "ent2id.json")
out_file_id2rel = os.path.join(output_dir, "id2rel.json")
out_file_rel2id = os.path.join(output_dir, "rel2id.json")

with open(out_file_id2ent, 'w') as file:
    json.dump(cpd_id2ent, file, indent=6)
with open(out_file_ent2id, 'w') as file:
    json.dump(cpd_ent2id, file, indent=6)
with open(out_file_id2rel, 'w') as file:
    json.dump(cpd_id2rel, file, indent=6)
with open(out_file_rel2id, 'w') as file:
    json.dump(cpd_rel2id, file, indent=6)

print(f"✓ Saved mapping files to {output_dir}")

In [ ]:
import numpy as np
import pickle

cond_ent_emb_file = os.path.join(base_dir, "graphs", "condition", "CCSCM", "entity_embedding.pkl")
cond_rel_emb_file = os.path.join(base_dir, "graphs", "condition", "CCSCM", "relation_embedding.pkl")
proc_ent_emb_file = os.path.join(base_dir, "graphs", "procedure", "CCSPROC", "entity_embedding.pkl")
proc_rel_emb_file = os.path.join(base_dir, "graphs", "procedure", "CCSPROC", "relation_embedding.pkl")
drug_ent_emb_file = os.path.join(base_dir, "graphs", "drug", "ATC3", "entity_embedding.pkl")
drug_rel_emb_file = os.path.join(base_dir, "graphs", "drug", "ATC3", "relation_embedding.pkl")

with open(cond_ent_emb_file, 'rb') as f:
    cond_ent_emb = pickle.load(f)

with open(cond_rel_emb_file, 'rb') as f:
    cond_rel_emb = pickle.load(f)

with open(proc_ent_emb_file, 'rb') as f:
    proc_ent_emb = pickle.load(f)

with open(proc_rel_emb_file, 'rb') as f:
    proc_rel_emb = pickle.load(f)

with open(drug_ent_emb_file, 'rb') as f:
    drug_ent_emb = pickle.load(f)

with open(drug_rel_emb_file, 'rb') as f:
    drug_rel_emb = pickle.load(f)

In [7]:
cond_ent_emb.shape, proc_ent_emb.shape, drug_ent_emb.shape

((10845, 1536), (8502, 1536), (22709, 1536))

In [8]:
# cp_ent_emb = np.concatenate((cond_ent_emb, proc_ent_emb), axis=0)
# cpd_ent_emb = np.concatenate((cp_ent_emb, drug_ent_emb), axis=0)

cp_rel_emb = np.concatenate((cond_rel_emb, proc_rel_emb), axis=0)
cpd_rel_emb = np.concatenate((cp_rel_emb, drug_rel_emb), axis=0)

In [9]:
cp_ent_emb.shape, cpd_ent_emb.shape

((19347, 1536), (42056, 1536))

In [ ]:
ent_emb_pkl = os.path.join(output_dir, "entity_embedding.pkl")
rel_emb_pkl = os.path.join(output_dir, "relation_embedding.pkl")

with open(ent_emb_pkl, "wb") as file:
    pickle.dump(cpd_ent_emb, file)

with open(rel_emb_pkl, "wb") as file:
    pickle.dump(cpd_rel_emb, file)

print(f"✓ Saved entity embeddings to {ent_emb_pkl}")
print(f"✓ Saved relation embeddings to {rel_emb_pkl}")
print(f"\nMerge complete! All files saved to: {output_dir}")